# FedGNN Baseline — H&M Sampled Dataset

Federated Graph Neural Network: a two-layer bipartite GNN where user and item
embeddings are updated via neighbourhood aggregation. Item embeddings (shared
global parameter) are aggregated across clients each round; user embeddings
stay local.


In [2]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import math
import os
import itertools

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


Device: cpu


In [3]:
from os import chdir
from pathlib import Path

new_path = Path("/Users/haowen/Documents/Decentral RS/fed-learning-main")

if new_path.exists():
    os.chdir(new_path)
    print(f"Working directory changed to: {Path.cwd()}")
else:
    print("Path does not exist.")


Working directory changed to: /Users/haowen/Documents/Decentral RS/fed-learning-main


In [4]:
from src.models.MatrixFactorization import MF, UMF
from src.graphs import random_k_out_graph, create_graph
from src.users import User
from src.training.decentralized import (decentralized_train_n_epochs,
                                        decentralized_validate_loop)
from src.data_utils import create_batched_dataloaders, create_dataloader

In [7]:
#Make data sample iterable
from torch.utils.data import Dataset, DataLoader, TensorDataset, Sampler
import torch
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.optim import SGD

from collections import Counter
import networkx as nx
from networkx.generators.classic import empty_graph
from networkx.utils import discrete_sequence, py_random_state, weighted_choice

import seaborn as sns
from sklearn.utils import shuffle

## 1. Load Data


In [9]:
SAMPLED_DATA_DIR = r"/Users/haowen/Documents/Decentral RS/rebuttal/code/8"

train_path = os.path.join(SAMPLED_DATA_DIR, "hm_subset_train.csv")
test_path  = os.path.join(SAMPLED_DATA_DIR, "hm_subset_test.csv")

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

# Cast IDs to str to handle mixed-type H&M IDs consistently
for df in [train_df, test_df]:
    df["user_id"] = df["user_id"].astype(str)
    df["item_id"] = df["item_id"].astype(str)

# Build contiguous 0-based encoders fitted on train only
user_enc = {u: i for i, u in enumerate(sorted(train_df["user_id"].unique()))}
item_enc = {v: i for i, v in enumerate(sorted(train_df["item_id"].unique()))}

def remap(df, user_enc, item_enc, drop_unknown=False):
    df = df.copy()
    if drop_unknown:
        df = df[df["user_id"].isin(user_enc) & df["item_id"].isin(item_enc)]
    df["user_id"] = df["user_id"].map(user_enc)
    df["item_id"] = df["item_id"].map(item_enc)
    return df.dropna(subset=["user_id", "item_id"]).astype({"user_id": int, "item_id": int})

train_df = remap(train_df, user_enc, item_enc)
test_df  = remap(test_df,  user_enc, item_enc, drop_unknown=True)

train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=0)
train_df.head()


,user_id,item_id,bought
17812,23,2123,1
788,491,714,1
1399,357,1741,1
12060,590,1828,1
31118,1248,1025,1


In [10]:
# Size from encoder fitted on full train — not from split subsets
n_users = len(user_enc)
n_items = len(item_enc)
print(f"Total Users: {n_users}")
print(f"Total Items: {n_items}")


Total Users: 1760
Total Items: 2881


## 2. Build Interaction Tensors

GNN operates on sparse (user, item) edge lists rather than dense matrices.


In [13]:
min_rating = float(train_df["bought"].min())
max_rating = float(train_df["bought"].max())

def df_to_edges(df, min_r, max_r):
    users = torch.tensor(df["user_id"].values, dtype=torch.long)
    items = torch.tensor(df["item_id"].values, dtype=torch.long)
    vals  = torch.tensor((df["bought"].values - min_r) / (max_r - min_r),
                         dtype=torch.float32)
    return users, items, vals

train_u, train_i, train_y = df_to_edges(train_df, min_rating, max_rating)
val_u,   val_i,   val_y   = df_to_edges(val_df,   min_rating, max_rating)
test_u,  test_i,  test_y  = df_to_edges(test_df,  min_rating, max_rating)

# Dense matrices for RMSE evaluation
def make_matrix(df, n_u, n_i, min_r, max_r):
    mat   = torch.full((n_u, n_i), -1.0)
    users = torch.tensor(df["user_id"].values, dtype=torch.long)
    items = torch.tensor(df["item_id"].values, dtype=torch.long)
    vals  = torch.tensor((df["bought"].values - min_r) / (max_r - min_r),
                         dtype=torch.float32)
    # Guard: drop any rows whose indices fall outside the embedding dimensions
    mask  = (users >= 0) & (users < n_u) & (items >= 0) & (items < n_i)
    if (~mask).any():
        print(f"  [make_matrix] dropping {(~mask).sum().item()} out-of-bounds rows")
    mat[users[mask], items[mask]] = vals[mask]
    return mat


In [15]:
train_matrix = make_matrix(train_df, n_users, n_items, min_rating, max_rating)
val_matrix   = make_matrix(val_df,   n_users, n_items, min_rating, max_rating)
test_matrix  = make_matrix(test_df,  n_users, n_items, min_rating, max_rating)
print(f"Train edges: {len(train_u)} | Val: {len(val_u)} | Test: {len(test_u)}")


Train edges: 62568 | Val: 15643 | Test: 19553


## 3. FedGNN Model (Wu et al., 2021)

Faithful implementation:
- **Per-user local subgraph**: each client owns only its own interactions.
- **GAT aggregation**: graph attention over first-order item neighbours.
- **Federated rounds**: server selects subset of clients, aggregates item/GAT gradients.
- **Pseudo-item sampling**: M fake item gradients mask real interaction items.
- **Local Differential Privacy**: L-inf clipping + Laplace noise on uploaded gradients.


In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math, random

# ── 1. GAT aggregation layer ─────────────────────────────────────────────────
class LocalGAT(nn.Module):
    """Single-head graph attention: user emb x item neighbourhood -> h_u."""
    def __init__(self, K):
        super().__init__()
        self.a = nn.Linear(2 * K, 1, bias=False)

    def forward(self, e_u, E_i):
        # e_u: (K,)   E_i: (n, K)
        n = E_i.size(0)
        if n == 0:
            return e_u
        e_u_exp = e_u.unsqueeze(0).expand(n, -1)          # (n, K)
        alpha   = F.softmax(self.a(torch.cat([e_u_exp, E_i], dim=1)), dim=0)  # (n,1)
        return F.relu((alpha * E_i).sum(dim=0))            # (K,)


# ── 2. Per-user client ────────────────────────────────────────────────────────
class UserClient(nn.Module):
    """
    Holds private user embedding e_u and references to shared item_emb_weight + gat.
    e_u is never uploaded to the server.
    """
    def __init__(self, user_id, K, item_emb_weight, gat):
        super().__init__()
        self.user_id         = user_id
        self.e_u             = nn.Parameter(torch.empty(K).normal_(std=0.01))
        self.item_emb_weight = item_emb_weight   # shared reference
        self.gat             = gat               # shared reference

    def forward(self, item_ids):
        """item_ids: LongTensor -> predicted ratings (len,)"""
        E_i   = self.item_emb_weight[item_ids]             # (n, K)
        h_u   = self.gat(self.e_u, E_i)                   # (K,)
        return torch.sigmoid((h_u.unsqueeze(0) * E_i).sum(dim=1))


# ── 3. Server (holds shared parameters) ──────────────────────────────────────
class FedGNNServer(nn.Module):
    def __init__(self, n_items, K):
        super().__init__()
        self.item_emb = nn.Embedding(n_items, K)
        self.gat      = LocalGAT(K)
        nn.init.normal_(self.item_emb.weight, std=0.01)

    def make_client(self, user_id):
        return UserClient(user_id,
                          self.item_emb.weight.size(1),
                          self.item_emb.weight,
                          self.gat)


# ── 4. LDP + pseudo item gradients ───────────────────────────────────────────
def apply_ldp(grad_tensor, delta=0.1, lam=0.2):
    """L-inf clip then Laplace(0, lam) noise."""
    clipped = grad_tensor.clamp(-delta, delta)
    noise   = torch.zeros_like(clipped).exponential_(1.0 / lam)
    noise  *= (2 * (torch.rand_like(clipped) > 0.5).float() - 1)
    return clipped + noise

def pseudo_item_gradients(real_grads, n_pseudo, n_items, K, device):
    """Generate M pseudo item gradients with same mean/std as real ones."""
    if real_grads.numel() == 0:
        mean = torch.zeros(K, device=device)
        std  = torch.ones(K, device=device) * 0.01
    else:
        mean = real_grads.mean(0)
        std  = real_grads.std(0).clamp(min=1e-6)
    pseudo_ids  = torch.randint(0, n_items, (n_pseudo,), device=device)
    pseudo_grad = torch.randn(n_pseudo, K, device=device) * std + mean
    return pseudo_ids, pseudo_grad


# ── 5. Evaluation helpers ─────────────────────────────────────────────────────
def compute_rmse(matrix, predicted, min_r, max_r):
    mask    = (matrix != -1).float()
    n       = mask.sum()
    if n == 0:
        return float('nan')
    pred_sc = predicted * (max_r - min_r) + min_r
    true_sc = matrix    * (max_r - min_r) + min_r
    return math.sqrt((((pred_sc - true_sc) ** 2 * mask).sum() / n).item())

def predict_all_global(server, user_clients, n_users, n_items,
                       user_items_dict, device):
    """Full (n_users, n_items) prediction matrix."""
    P = torch.zeros(n_users, n_items, device=device)
    server.eval()
    with torch.no_grad():
        item_w = server.item_emb.weight          # (n_items, K)
        for uid, client in user_clients.items():
            items = user_items_dict.get(uid)
            if not items:
                continue
            items_t = torch.tensor(items, dtype=torch.long, device=device)
            E_i     = item_w[items_t]
            h_u     = server.gat(client.e_u.detach(), E_i)
            P[uid]  = torch.sigmoid((h_u.unsqueeze(0) * item_w).sum(dim=1))
    return P

print("FedGNN (Wu et al. 2021) classes defined.")


FedGNN (Wu et al. 2021) classes defined.


## 4. Build Per-User Data Structures & Hyperparameters


In [24]:
# ── Hyperparameters (Wu et al. 2021 defaults, scaled for H&M) ────────────────
K           = 64      # embedding dim (paper: 256; reduce for speed)
LR          = 0.01    # SGD lr
LAM         = 0.01    # L2 regularisation
DELTA       = 0.1     # LDP clip threshold
LAMBDA_LDP  = 0.2     # LDP Laplace noise strength  (achieves 1-diff-privacy)
M_PSEUDO    = 200     # pseudo item count per client per round
N_ROUNDS    = 100     # total federated communication rounds
CLIENT_FRAC = 0.1     # fraction of clients sampled each round
PATIENCE    = 10      # early stopping on val RMSE (checked every 5 rounds)
BATCH_SIZE  = 32      # interactions sampled per client per round

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ── Per-user interaction lookup (train only) ──────────────────────────────────
user_items_dict   = {}   # uid -> [item_ids]
user_ratings_dict = {}   # uid -> {item_id: normalised_rating}
denom = max(max_rating - min_rating, 1e-8)

for row in train_df.itertuples(index=False):
    uid = int(row.user_id)
    iid = int(row.item_id)
    val = float((row.bought - min_rating) / denom)
    user_items_dict.setdefault(uid, []).append(iid)
    user_ratings_dict.setdefault(uid, {})[iid] = val

all_user_ids = sorted(user_items_dict.keys())
print(f"Clients with training data : {len(all_user_ids)}")
print(f"Items in embedding table   : {n_items}")


Device: cpu
Clients with training data : 1760
Items in embedding table   : 2881


## 5. Parameter Tuning — Grid Search

Grid over `lr`, `lam` (L2 regularisation), and `K` (embedding dim).
Runs a lightweight federated simulation (reduced clients + epochs) on the
**validation matrix** and writes `LR`, `LAM`, `K` for the full training run.


In [36]:
# ── Grid ─────────────────────────────────────────────────────────────────────
LR_GRID     = [0.001, 0.01,  0.05]
LAM_GRID    = [0.01,  0.1,   0.3 ]
K_GRID      = [16,    32,    64  ]
TUNE_ROUNDS = 20       # federated rounds per combo
TUNE_PAT    = 10        # early-stop patience (checked every round)
TUNE_FRAC   = 0.2     # client fraction for tuning (faster than full 0.1)

param_grid = list(itertools.product(LR_GRID, LAM_GRID, K_GRID))
print(f"Grid search: {len(param_grid)} combinations x up to {TUNE_ROUNDS} rounds")
print(f"{'#':>3}  {'lr':>7}  {'lam':>6}  {'K':>4}  {'best val RMSE':>14}")
print("-" * 42)

grid_results = []

for _k, (_lr, _lam, _K) in enumerate(param_grid, 1):
    torch.manual_seed(0)
    random.seed(0)

    # Init a fresh server + clients for this combo
    _server = FedGNNServer(n_items, _K).to(device)
    _clients = {}
    for uid in all_user_ids:
        c = _server.make_client(uid)
        c.e_u = nn.Parameter(c.e_u.to(device))
        _clients[uid] = c

    _srv_opt = torch.optim.SGD(
        list(_server.item_emb.parameters()) + list(_server.gat.parameters()),
        lr=_lr, weight_decay=_lam)

    _n_cli   = max(1, int(len(all_user_ids) * TUNE_FRAC))
    _best_v, _wait = float('inf'), 0

    for _rnd in range(1, TUNE_ROUNDS + 1):
        _sampled = random.sample(all_user_ids, _n_cli)
        _acc_ig  = torch.zeros_like(_server.item_emb.weight)
        _acc_gg  = [torch.zeros_like(p) for p in _server.gat.parameters()]

        _server.train()
        for uid in _sampled:
            _cl   = _clients[uid]
            _ials = user_items_dict[uid]
            if not _ials:
                continue
            _bidx   = random.sample(range(len(_ials)), min(BATCH_SIZE, len(_ials)))
            _bitems = [_ials[j] for j in _bidx]
            _it     = torch.tensor(_bitems, dtype=torch.long, device=device)
            _tgt    = torch.tensor([user_ratings_dict[uid][i] for i in _bitems],
                                   dtype=torch.float32, device=device)
            if _cl.e_u.grad is not None: _cl.e_u.grad.zero_()
            _srv_opt.zero_grad()
            _pred = _cl(_it)
            _loss = (F.mse_loss(_pred, _tgt)
                     + _lam * (_cl.e_u.norm()
                                + _server.item_emb(_it).norm()
                                + sum(p.norm() for p in _server.gat.parameters())))
            _loss.backward()
            with torch.no_grad():
                if _cl.e_u.grad is not None:
                    _cl.e_u.data -= _lr * _cl.e_u.grad
            with torch.no_grad():
                _rig = _server.item_emb.weight.grad[_it].clone()
                _pid, _pg = pseudo_item_gradients(_rig, M_PSEUDO, n_items, _K, device)
                _acc_ig.index_add_(0, _it,  apply_ldp(_rig, DELTA, LAMBDA_LDP))
                _acc_ig.index_add_(0, _pid, _pg)
                for _ag, _p in zip(_acc_gg, _server.gat.parameters()):
                    if _p.grad is not None:
                        _ag.add_(apply_ldp(_p.grad.clone(), DELTA, LAMBDA_LDP))

        with torch.no_grad():
            _server.item_emb.weight.grad = _acc_ig / _n_cli
            for _p, _ag in zip(_server.gat.parameters(), _acc_gg):
                _p.grad = _ag / _n_cli
        _srv_opt.step()

        # Val RMSE
        _server.eval()
        with torch.no_grad():
            _pm  = predict_all_global(_server, _clients, n_users, n_items,
                                      user_items_dict, device)
            _v   = compute_rmse(val_matrix.to(device), _pm, min_rating, max_rating)

        if _v < _best_v:
            _best_v, _wait = _v, 0
        else:
            _wait += 1
            if _wait >= TUNE_PAT:
                break

    grid_results.append((_best_v, _lr, _lam, _K))
    _marker = ' <--' if _best_v == min(r[0] for r in grid_results) else ''
    print(f"{_k:>3}  {_lr:>7.4f}  {_lam:>6.3f}  {_K:>4d}  {_best_v:>14.4f}{_marker}")

_best = min(grid_results, key=lambda x: x[0])
best_val_rmse_tune, best_lr, best_lam, best_K = _best
print(f"\nBest val RMSE  : {best_val_rmse_tune:.4f}")
print(f"  LR             = {best_lr}")
print(f"  LAM            = {best_lam}")
print(f"  K              = {best_K}")
LR  = best_lr
LAM = best_lam
K   = best_K
print("\nHyperparameters updated. Run the training cell.")


Grid search: 27 combinations x up to 20 rounds
  #       lr     lam     K   best val RMSE
------------------------------------------
  1   0.0010   0.010    16         11.0900 <--
  2   0.0010   0.010    32         11.0900
  3   0.0010   0.010    64         11.0900
  4   0.0010   0.100    16         11.0900
  5   0.0010   0.100    32         11.0900
  6   0.0010   0.100    64         11.0900
  7   0.0010   0.300    16         11.0900 <--
  8   0.0010   0.300    32         11.0900
  9   0.0010   0.300    64         11.0900
 10   0.0100   0.010    16         11.0900 <--
 11   0.0100   0.010    32         11.0900
 12   0.0100   0.010    64         11.0900
 13   0.0100   0.100    16         11.0900
 14   0.0100   0.100    32         11.0900
 15   0.0100   0.100    64         11.0900
 16   0.0100   0.300    16         11.0900 <--
 17   0.0100   0.300    32         11.0900
 18   0.0100   0.300    64         11.0900
 19   0.0500   0.010    16         11.0900 <--
 20   0.0500   0.010    32    

## 6. Federated Training


In [39]:
# ── Use tuned hyperparameters (fall back to defaults if grid search skipped) ─
try: K
except NameError: K = 64
try: LR
except NameError: LR = 0.01
try: LAM
except NameError: LAM = 0.01

torch.manual_seed(0)
random.seed(0)

# ── Init server + per-user clients ───────────────────────────────────────────
server = FedGNNServer(n_items, K).to(device)
user_clients = {}
for uid in all_user_ids:
    c      = server.make_client(uid)
    c.e_u  = nn.Parameter(c.e_u.to(device))
    user_clients[uid] = c

server_opt = torch.optim.SGD(
    list(server.item_emb.parameters()) + list(server.gat.parameters()),
    lr=LR, weight_decay=LAM
)

best_val_rmse    = float('inf')
patience_count   = 0
best_item_state  = None
best_gat_state   = None
best_user_states = {}

n_cli = max(1, int(len(all_user_ids) * CLIENT_FRAC))

for rnd in range(1, N_ROUNDS + 1):

    # 1. Sample clients
    sampled = random.sample(all_user_ids, n_cli)

    accum_item_grad = torch.zeros_like(server.item_emb.weight)
    accum_gat_grads = [torch.zeros_like(p) for p in server.gat.parameters()]

    server.train()

    for uid in sampled:
        client    = user_clients[uid]
        items_all = user_items_dict[uid]
        if not items_all:
            continue

        # 2. Mini-batch
        bidx     = random.sample(range(len(items_all)),
                                 min(BATCH_SIZE, len(items_all)))
        bitems   = [items_all[j] for j in bidx]
        items_t  = torch.tensor(bitems, dtype=torch.long, device=device)
        targets  = torch.tensor(
            [user_ratings_dict[uid][i] for i in bitems],
            dtype=torch.float32, device=device)

        # 3. Forward + loss
        if client.e_u.grad is not None:
            client.e_u.grad.zero_()
        server_opt.zero_grad()

        preds = client(items_t)
        loss  = (F.mse_loss(preds, targets)
                 + LAM * (client.e_u.norm()
                          + server.item_emb(items_t).norm()
                          + sum(p.norm() for p in server.gat.parameters())))
        loss.backward()

        # 4. Update private user embedding locally (never uploaded)
        with torch.no_grad():
            if client.e_u.grad is not None:
                client.e_u.data -= LR * client.e_u.grad

        # 5. Collect & protect item + GAT gradients for upload
        with torch.no_grad():
            real_ig = server.item_emb.weight.grad[items_t].clone()

            # Pseudo item gradients (privacy masking)
            p_ids, p_grads = pseudo_item_gradients(
                real_ig, M_PSEUDO, n_items, K, device)

            # LDP on real gradients
            noisy_real = apply_ldp(real_ig, DELTA, LAMBDA_LDP)

            accum_item_grad.index_add_(0, items_t, noisy_real)
            accum_item_grad.index_add_(0, p_ids,   p_grads)

            for acc, p in zip(accum_gat_grads, server.gat.parameters()):
                if p.grad is not None:
                    acc.add_(apply_ldp(p.grad.clone(), DELTA, LAMBDA_LDP))

    # 6. Server aggregates + updates shared params
    with torch.no_grad():
        server.item_emb.weight.grad = accum_item_grad / n_cli
        for p, acc in zip(server.gat.parameters(), accum_gat_grads):
            p.grad = acc / n_cli
    server_opt.step()

    # 7. Validation every 5 rounds
    if rnd % 5 == 0 or rnd == 1:
        server.eval()
        with torch.no_grad():
            pred_mat   = predict_all_global(server, user_clients, n_users,
                                            n_items, user_items_dict, device)
            train_rmse = compute_rmse(train_matrix.to(device), pred_mat,
                                      min_rating, max_rating)
            val_rmse   = compute_rmse(val_matrix.to(device),   pred_mat,
                                      min_rating, max_rating)
        print(f"Round {rnd:4d} | train RMSE: {train_rmse:.4f} "
              f"| val RMSE: {val_rmse:.4f}")

        if val_rmse < best_val_rmse:
            best_val_rmse    = val_rmse
            best_item_state  = server.item_emb.state_dict()
            best_gat_state   = server.gat.state_dict()
            best_user_states = {uid: c.e_u.detach().clone()
                                for uid, c in user_clients.items()}
            patience_count   = 0
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print(f"Early stopping at round {rnd}.")
                break

# Restore best checkpoint
server.item_emb.load_state_dict(best_item_state)
server.gat.load_state_dict(best_gat_state)
for uid, eu in best_user_states.items():
    user_clients[uid].e_u.data.copy_(eu)

print(f"\nBest val RMSE: {best_val_rmse:.4f}")


Round    1 | train RMSE: 11.1029 | val RMSE: 11.0900
Round    5 | train RMSE: 11.1029 | val RMSE: 11.0900
Round   10 | train RMSE: 11.1029 | val RMSE: 11.0900
Round   15 | train RMSE: 11.1029 | val RMSE: 11.0900
Round   20 | train RMSE: 11.1029 | val RMSE: 11.0900
Round   25 | train RMSE: 11.1029 | val RMSE: 11.0900
Round   30 | train RMSE: 11.1029 | val RMSE: 11.0900
Round   35 | train RMSE: 11.1029 | val RMSE: 11.0900
Round   40 | train RMSE: 11.1029 | val RMSE: 11.0900
Round   45 | train RMSE: 11.1029 | val RMSE: 11.0900
Round   50 | train RMSE: 11.1029 | val RMSE: 11.0900
Early stopping at round 50.

Best val RMSE: 11.0900


## 7. Test Evaluation


In [42]:
server.eval()
with torch.no_grad():
    pred_test     = predict_all_global(server, user_clients, n_users, n_items,
                                       user_items_dict, device)
    pred_test_cpu = pred_test.cpu()
    test_rmse     = compute_rmse(test_matrix, pred_test_cpu, min_rating, max_rating)

    mask     = (test_matrix != -1).float()
    n_obs    = mask.sum()
    pred_sc  = pred_test_cpu * (max_rating - min_rating) + min_rating
    true_sc  = test_matrix   * (max_rating - min_rating) + min_rating
    test_mae = (torch.abs(pred_sc - true_sc) * mask).sum() / n_obs

print(f"Test RMSE : {test_rmse:.4f}")
print(f"Test MAE  : {test_mae.item():.4f}")


Test RMSE : 11.1019
Test MAE  : 11.0611
